In [47]:
import pandas as pd
from pathlib import Path

NOTE:
Raw TXT ingestion code is not executed here for performance.
See commented block below for full reproducible pipeline.
The analysis uses the pre-cleaned CSV stored in the repo:
https://raw.githubusercontent.com/MIDS-at-Duke/pds-2025-opioids-straight-as/refs/heads/Rosella/00_data/clean/vital_stats_2003_2015.csv?token=GHSAT0AAAAAADPWOMOTQLLGJB7AKHY64TO22J3I7IQ

In [48]:
# Where raw txt files were stored
# RAW_PATH = "/Users/rosellazheng/Desktop/US_VitalStatistics/"
# pattern = RAW_PATH + "Underlying Cause of Death, 20*.txt"

# files = glob.glob(pattern)
# print("Found files:", files)  # All are in there

# Combine all years
# df = pd.concat((pd.read_csv(f, sep="\t") for f in files), ignore_index=True)

# Quick clean of column names
# df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Save into project repo so everyone else can load this CSV
# OUTPUT = "/Users/rosellazheng/Desktop/idsproject/vital_stats_2003_2015.csv"
# df.to_csv(OUTPUT, index=False)

# print("Saved clean CSV to:", OUTPUT)

In [49]:
OUTPUT_URL = "https://raw.githubusercontent.com/MIDS-at-Duke/pds-2025-opioids-straight-as/refs/heads/Rosella/00_data/clean/vital_stats_2003_2015.csv?token=GHSAT0AAAAAADPWOMOTQLLGJB7AKHY64TO22J3I7IQ"

In [50]:
# Load the data once
df = pd.read_csv(OUTPUT_URL, dtype=str)
print(f"Raw data shape: {df.shape}")  # e.g., (57436, 8)

Raw data shape: (57436, 8)


In [51]:
# Quick look at columns (optional, can be commented out)
print(df.columns.unique())

Index(['notes', 'county', 'county_code', 'year', 'year_code',
       'drug/alcohol_induced_cause', 'drug/alcohol_induced_cause_code',
       'deaths'],
      dtype='object')


In [52]:
# --------------------------------------------------------------------
# 1. Drop rows with non-NA notes (these are the "weird notes" rows)
# --------------------------------------------------------------------
df_clean = df[df["notes"].isna()].copy()  # .copy() avoids SettingWithCopyWarning
print("Rows after dropping notes:", len(df_clean))  # expected: 57241
assert len(df_clean) == 57241, f"Row count mismatch after notes filter: {len(df_clean)}"

Rows after dropping notes: 57241


In [53]:
# Drop notes and year_code columns
df_clean = df_clean.drop(columns=["notes", "year_code"])
print(df_clean.columns)
print(df_clean.head())

Index(['county', 'county_code', 'year', 'drug/alcohol_induced_cause',
       'drug/alcohol_induced_cause_code', 'deaths'],
      dtype='object')
               county county_code    year  \
0  Autauga County, AL      1001.0  2009.0   
1  Baldwin County, AL      1003.0  2009.0   
2  Baldwin County, AL      1003.0  2009.0   
3  Barbour County, AL      1005.0  2009.0   
4     Bibb County, AL      1007.0  2009.0   

                          drug/alcohol_induced_cause  \
0          All other non-drug and non-alcohol causes   
1  Drug poisonings (overdose) Unintentional (X40-...   
2          All other non-drug and non-alcohol causes   
3          All other non-drug and non-alcohol causes   
4          All other non-drug and non-alcohol causes   

  drug/alcohol_induced_cause_code  deaths  
0                              O9   408.0  
1                              D1    29.0  
2                              O9  1669.0  
3                              O9   299.0  
4                          

In [54]:
# ---------------------------------------------------
# 2. Split county into county + state
# ---------------------------------------------------
df_clean[["county", "state"]] = df_clean["county"].str.split(", ", expand=True)
# print(df_clean.head())
print("Rows after county/state split:", len(df_clean))  # sanity check

Rows after county/state split: 57241


In [55]:
# here we can change both county_code and year into int
df_clean.dtypes

county                             object
county_code                        object
year                               object
drug/alcohol_induced_cause         object
drug/alcohol_induced_cause_code    object
deaths                             object
state                              object
dtype: object

In [56]:
# ---------------------------------------------------
# 3. Check categories for drug/alcohol induced cause
# ---------------------------------------------------
print(df_clean["drug/alcohol_induced_cause"].unique())

['All other non-drug and non-alcohol causes'
 'Drug poisonings (overdose) Unintentional (X40-X44)'
 'Drug poisonings (overdose) Undetermined (Y10-Y14)'
 'Drug poisonings (overdose) Suicide (X60-X64)'
 'All other drug-induced causes' 'All other alcohol-induced causes'
 'Alcohol poisonings (overdose) (X45, X65, Y15)'
 'Drug poisonings (overdose) Homicide (X85)']


In [57]:
# Keep only unintentional drug poisonings (X40–X44)
df_overdose = df_clean[
    df_clean["drug/alcohol_induced_cause"]
    == "Drug poisonings (overdose) Unintentional (X40-X44)"
].copy()

In [58]:
df_overdose

,county,county_code,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,deaths,state
1,Baldwin County,1003.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,29.0,AL
8,Calhoun County,1015.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,12.0,AL
30,Etowah County,1055.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,AL
38,Houston County,1069.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,10.0,AL
41,Jefferson County,1073.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,80.0,AL
...,...,...,...,...,...,...,...
57379,Washington County,55131.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,19.0,WI
57382,Waukesha County,55133.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,39.0,WI
57388,Winnebago County,55139.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,WI
57403,Laramie County,56021.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,15.0,WY


In [59]:
print(
    "Rows after filtering unintentional overdoses:", len(df_overdose)
)  # expected: 7573
assert (
    len(df_overdose) == 7573
), f"Row count mismatch after overdose filter: {len(df_overdose)}"

Rows after filtering unintentional overdoses: 7573


In [60]:
# Confirm only D1 in cause code
print(df_overdose["drug/alcohol_induced_cause_code"].unique())

['D1']


In [61]:
# ---------------------------------------------------
# 4. Exclude Alaska
# ---------------------------------------------------
df_overdose = df_overdose[df_overdose["state"] != "AK"].copy()
print("Rows after dropping AK:", len(df_overdose))  # expected: 7538
assert (
    len(df_overdose) == 7538
), f"Row count mismatch after dropping AK: {len(df_overdose)}"

Rows after dropping AK: 7538


In [62]:
df_overdose

,county,county_code,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,deaths,state
1,Baldwin County,1003.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,29.0,AL
8,Calhoun County,1015.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,12.0,AL
30,Etowah County,1055.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,AL
38,Houston County,1069.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,10.0,AL
41,Jefferson County,1073.0,2009.0,Drug poisonings (overdose) Unintentional (X40-...,D1,80.0,AL
...,...,...,...,...,...,...,...
57379,Washington County,55131.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,19.0,WI
57382,Waukesha County,55133.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,39.0,WI
57388,Winnebago County,55139.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,WI
57403,Laramie County,56021.0,2013.0,Drug poisonings (overdose) Unintentional (X40-...,D1,15.0,WY


In [63]:
# ---------------------------------------------------
# 5. Fix dtypes
# ---------------------------------------------------
df_overdose["deaths"] = pd.to_numeric(df_overdose["deaths"], errors="coerce").astype(
    "Int64"
)

df_overdose["county"] = df_overdose["county"].astype("string")
df_overdose["county_code"] = pd.to_numeric(
    df_overdose["county_code"], errors="coerce"
).astype("Int64")

df_overdose["drug/alcohol_induced_cause"] = df_overdose[
    "drug/alcohol_induced_cause"
].astype("string")

df_overdose["drug/alcohol_induced_cause_code"] = df_overdose[
    "drug/alcohol_induced_cause_code"
].astype("string")

df_overdose["year"] = pd.to_numeric(df_overdose["year"], errors="coerce").astype(
    "Int64"
)
df_overdose["state"] = df_overdose["state"].astype("string")

print("Rows after dtype fixes:", len(df_overdose))  # should still be 7538
assert len(df_overdose) == 7538, "Row count changed during dtype conversion!"

Rows after dtype fixes: 7538


In [64]:
df_overdose

,county,county_code,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,deaths,state
1,Baldwin County,1003,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,29,AL
8,Calhoun County,1015,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,12,AL
30,Etowah County,1055,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,14,AL
38,Houston County,1069,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,10,AL
41,Jefferson County,1073,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,80,AL
...,...,...,...,...,...,...,...
57379,Washington County,55131,2013,Drug poisonings (overdose) Unintentional (X40-...,D1,19,WI
57382,Waukesha County,55133,2013,Drug poisonings (overdose) Unintentional (X40-...,D1,39,WI
57388,Winnebago County,55139,2013,Drug poisonings (overdose) Unintentional (X40-...,D1,14,WI
57403,Laramie County,56021,2013,Drug poisonings (overdose) Unintentional (X40-...,D1,15,WY


In [65]:
# Optional checks
print(df_overdose["drug/alcohol_induced_cause"].unique())
print(df_overdose.dtypes)

<StringArray>
['Drug poisonings (overdose) Unintentional (X40-X44)']
Length: 1, dtype: string
county                             string[python]
county_code                                 Int64
year                                        Int64
drug/alcohol_induced_cause         string[python]
drug/alcohol_induced_cause_code    string[python]
deaths                                      Int64
state                              string[python]
dtype: object


In [66]:
# ---------------------------------------------------
# 6. Restrict to overlapping years with teammate data
#    (2006–2015)
# ---------------------------------------------------
df_overdose = df_overdose[
    (df_overdose["year"] >= 2006) & (df_overdose["year"] <= 2015)
].copy()

print("Rows after restricting to 2006~2015:", len(df_overdose))  # expected: 6300
assert (
    len(df_overdose) == 6300
), f"Row count mismatch after year filter: {len(df_overdose)}"

Rows after restricting to 2006~2015: 6300


In [67]:
# Quick sanity check by state
print(df_overdose["state"].value_counts())
print(df_overdose["state"].value_counts().sum())

state
CA    383
NC    354
TX    353
OH    353
FL    349
PA    345
GA    262
TN    230
NY    216
KY    211
IN    202
VA    179
MI    178
NJ    175
IL    173
WA    157
SC    149
AL    147
LA    144
WV    143
MO    136
OK    127
WI    117
MA    111
CO    105
NM     96
AZ     86
OR     82
CT     74
MN     65
AR     61
MS     53
UT     48
NH     48
ME     47
KS     46
MD     45
RI     32
IA     32
NV     31
ID     30
DE     28
NE     20
HI     19
MT     16
WY     15
VT     12
DC     10
SD      4
ND      1
Name: count, dtype: Int64
6300


In [68]:
# ---------------------------------------------------
# 7. Save cleaned overdose dataset to project repo
# ---------------------------------------------------

# Save relative to project (not your Desktop)
OUTPUT2 = Path("00_data/clean/df_overdose_cleaned.csv")
OUTPUT2.parent.mkdir(parents=True, exist_ok=True)
df_overdose.to_csv(OUTPUT2, index=False)
print(f"Saved cleaned overdose CSV to: {OUTPUT2}")

Saved cleaned overdose CSV to: 00_data/clean/df_overdose_cleaned.csv
